In [11]:
import numpy as np
import pandas as pd
import glob, os
import matplotlib.pyplot as plt
import seaborn as sns
import scipy
import scipy.stats as st
import statsmodels.stats.api as sm

import Bio.PDB
from Bio import Seq, SeqIO
from Bio.PDB.MMCIFParser import MMCIFParser
from Bio.PDB.DSSP import make_dssp_dict
from Bio.PDB.Polypeptide import protein_letters_3to1

h37Rv = SeqIO.read("/n/data1/hms/dbmi/farhat/Sanjana/H37Rv/GCF_000195955.2_ASM19595v2_genomic.gbff", "genbank")
h37Rv_genes = pd.read_csv("/n/data1/hms/dbmi/farhat/Sanjana/H37Rv/mycobrowser_h37rv_genes_v4.csv")

# Make distance maps

## Notes for AlphaFold structures:

Code to save only the high confidence coordinates

```
select high_confidence, b > 70
save ethA_alphaFold_highConf.pdb, high_confidence
```

<!-- 
<ul>
    <li></li>
</ul> -->

In [2]:
# # need to leave the Unnamed: 0 index column (don't save with index = False) because evcouplings.compare.distances.py reads in the dataframe with index_col = 0
# pncA_structure_coords = np.load("distance_maps/I6XD65.npy")
# pncA_distance_map = pd.read_csv("distance_maps/I6XD65.csv")

# katG_structure_coords = np.load("distance_maps/P9WIE5.npy")
# katG_distance_map = pd.read_csv("distance_maps/P9WIE5.csv")

# # check that this is a pairwise matrix, meaning that it's symmetric
# assert scipy.linalg.issymmetric(pncA_structure_coords)
# assert scipy.linalg.issymmetric(katG_structure_coords)

# # and also that the diagonals are all 0
# assert sum(np.diagonal(pd.DataFrame(pncA_structure_coords))) == 0
# assert sum(np.diagonal(pd.DataFrame(katG_structure_coords))) == 0

In [3]:
# Function to parse CIF file and extract necessary information
def extract_cif_info(cif_file):
    parser = MMCIFParser()
    structure = parser.get_structure('protein', cif_file)
    
    model = structure[0]
    
    # List to hold extracted information
    data = []
    
    # Extract information for each residue
    for chain in model:
        chain_id = chain.id
        for i, res in enumerate(chain):
            if res.id[0] == ' ':  # Exclude heteroatoms for now
                res_id = res.id[1]
                seqres_id = i + 1
                res_name = res.resname
                try:
                    one_letter_code = protein_letters_3to1[res_name]
                except KeyError:
                    one_letter_code = 'X'  # Unknown residue
                
                hetatm = res.id[0] != ' '
                coord = res['CA'].coord if 'CA' in res else None

                # # Get secondary structure assignment from DSSP -- not necessary for spatial clustering, and need to install additional dependencies, so skip for now
                # dssp_key = (chain_id, (' ', res_id, ' '))
                # if dssp_key in dssp_dict:
                #     sec_struct = dssp_dict[dssp_key][1]
                #     sec_struct_3state = 'H' if sec_struct in 'GHI' else 'E' if sec_struct == 'E' else 'C'
                # else:
                #     sec_struct = 'NA'
                #     sec_struct_3state = 'NA'
                sec_struct = 'NA'
                sec_struct_3state = 'NA'

                # chain index = 0 because there is only one chain
                # add 1 to len(data) to make it 1-indexed (in residue coordinate space, not index)
                data.append([
                    len(data) + 1, seqres_id, res_id, one_letter_code, res_name,
                    0, chain_id, sec_struct, sec_struct_3state, hetatm, coord
                ])
    
    # Create DataFrame
    columns = ['id', 'seqres_id', 'coord_id', 'one_letter_code',
               'three_letter_code', 'chain_index', 'chain_id', 'sec_struct',
               'sec_struct_3state', 'hetatm', 'coord']
    df = pd.DataFrame(data, columns=columns)
    return df

In [30]:
# mmcif_file = 'ethA_AlphaFold.cif'

# df_ethA_AF = extract_cif_info(mmcif_file)

# # residues 1 and 484-489 are low-confidence in alpha fold, so exclude
# df_ethA_AF_highConf = df_ethA_AF.query("id >= 2 & id <= 483").reset_index(drop=True)
# df_ethA_AF_highConf.to_csv("distance_maps/P9WNF9_AF.csv")

In [67]:
RNA_polymerase = extract_cif_info("5zx3.cif")
gyrA_N_terminus = extract_cif_info("gyrA_N_terminus_3ifz.cif")

/home/sak0914/anaconda3/lib/python3.9/site-packages/Bio/PDB/StructureBuilder.py:89: PDBConstructionWarning: WARNING: Chain D is discontinuous at line 23256.
  warnings.warn(
/home/sak0914/anaconda3/lib/python3.9/site-packages/Bio/PDB/StructureBuilder.py:89: PDBConstructionWarning: WARNING: Chain A is discontinuous at line 23258.
  warnings.warn(
/home/sak0914/anaconda3/lib/python3.9/site-packages/Bio/PDB/StructureBuilder.py:89: PDBConstructionWarning: WARNING: Chain C is discontinuous at line 23259.
  warnings.warn(
/home/sak0914/anaconda3/lib/python3.9/site-packages/Bio/PDB/StructureBuilder.py:89: PDBConstructionWarning: WARNING: Chain D is discontinuous at line 23288.
  warnings.warn(
/home/sak0914/anaconda3/lib/python3.9/site-packages/Bio/PDB/StructureBuilder.py:89: PDBConstructionWarning: WARNING: Chain F is discontinuous at line 23310.
  warnings.warn(
/home/sak0914/anaconda3/lib/python3.9/site-packages/Bio/PDB/StructureBuilder.py:89: PDBConstructionWarning: WARNING: Chain A is di

In [82]:
# from PDB, chain C = rpoB, but the number is slightly off too
# residue 7 in PDB is residue 1 in H37Rv, so subtract 6
rpoB = RNA_polymerase.query("chain_id=='C'").reset_index(drop=True)
rpoB['coord_id'] -= 6
rpoB['id'] = rpoB['coord_id']

rpoB.to_csv("distance_maps/5ZX3_rpoB.csv")

# did some manual checks
rpoB_protein_seq = h37Rv.seq[759807-1:763325].translate()
gyrA_protein_seq = h37Rv.seq[7301:9818].translate()

assert rpoB_protein_seq[-1] == '*'
assert gyrA_protein_seq[-1] == '*'

gyrA_N_terminus['id'] = gyrA_N_terminus['coord_id']
gyrA_N_terminus.to_csv("distance_maps/3IFZ.csv")

In [30]:
def calculate_pairwise_distances(coords):
    coords = np.array([coord for coord in coords if coord is not None])
    distances = np.linalg.norm(coords[:, np.newaxis] - coords, axis=-1)
    return distances

In [86]:
def get_distance_map_coordinates(df, dm_name):
    
    df.to_csv(f"distance_maps/{dm_name}.csv")

    assert sum(pd.isnull(df['coord'])) == 0
    
    # the coordinates are for the alpha carbon atom in each residue
    ca_coords = list(df['coord'])
    ca_distances = calculate_pairwise_distances(ca_coords)
    
    np.save(f"distance_maps/{dm_name}.npy", ca_distances)
    
    # pairwise matrix check
    assert scipy.linalg.issymmetric(ca_distances)
    assert sum(np.diagonal(pd.DataFrame(ca_distances))) == 0
    
    print(ca_distances.shape)

In [88]:
generate_distance_map_from_CIF(gyrA_N_terminus, "3IFZ")

(487, 487)


/home/sak0914/anaconda3/lib/python3.9/site-packages/Bio/PDB/StructureBuilder.py:89: PDBConstructionWarning: WARNING: Chain A is discontinuous at line 7534.
  warnings.warn(
/home/sak0914/anaconda3/lib/python3.9/site-packages/Bio/PDB/StructureBuilder.py:89: PDBConstructionWarning: WARNING: Chain B is discontinuous at line 7543.
  warnings.warn(
/home/sak0914/anaconda3/lib/python3.9/site-packages/Bio/PDB/StructureBuilder.py:89: PDBConstructionWarning: WARNING: Chain A is discontinuous at line 7552.
  warnings.warn(
/home/sak0914/anaconda3/lib/python3.9/site-packages/Bio/PDB/StructureBuilder.py:89: PDBConstructionWarning: WARNING: Chain B is discontinuous at line 7656.
  warnings.warn(


# Results of Clustering on Averaged Fold Change MIC Predictions from Site-Saturation Mutagenesis

In [89]:
def get_significant_GeO_scores(results_dir, prefix, pval_thresh=0.05):

    if not os.path.isfile(f"{results_dir}/{prefix}_values_to_cluster.csv"):
        print("\nThere are no negative sites")
        return None
    
    residue_data = pd.read_csv(f"{results_dir}/{prefix}_values_to_cluster.csv")

    GeO_scores = pd.read_csv(f"{results_dir}/{prefix}_G_scores.csv").merge(residue_data, on='residue')
    
    # these are GeO scores for each residue after shuffling the residues
    # permutation test to see if the GeO score for each residue is significantly different from the the null
    GeO_permutation_results = pd.read_csv(f"{results_dir}/{prefix}_random_GeO_iterations_10000.csv.gz", compression="gzip", index_col=[0])
    
    # GeO_pvalues = pd.read_csv(f"{results_dir}/random_GeO_pvalues_10000.csv.gz", compression="gzip")

    for i, row in GeO_scores.iterrows():
    
        residue = row['residue']
        GeO_score = row['G_score']
    
        # what proportion of the permuted Getis-Ord statistics are at least as extreme as the Getis-Ord statistic for a given residue
        if GeO_score > 0:
            pvalue = np.mean(GeO_permutation_results.loc[residue, :].values >= GeO_score)
        else:
            pvalue = np.mean(GeO_permutation_results.loc[residue, :].values <= GeO_score)
    
        GeO_scores.loc[i, "pval"] = pvalue

    _, bh_pvals, _, _ = sm.multipletests(GeO_scores["pval"], method='fdr_bh', is_sorted=False, returnsorted=False)
    _, bonferroni_pvals, _, _ = sm.multipletests(GeO_scores["pval"], method='bonferroni', is_sorted=False, returnsorted=False)
    
    GeO_scores['BH_pval'] = bh_pvals
    GeO_scores['Bonferroni_pval'] = bonferroni_pvals
    GeO_scores['prefix'] = prefix
    
    hot_spots = GeO_scores.query("BH_pval <= @pval_thresh & G_score > 0")
    cold_spots = GeO_scores.query("BH_pval <= @pval_thresh & G_score < 0")

    print(f"\n{len(hot_spots)} hot spot residues")
    print(f"{len(cold_spots)} cold spot residues")

    return GeO_scores

In [60]:
catalytic_triad = [8, 96, 138]
iron_coordinating = [49, 51, 57, 71]

results_dir = "pncA"

pncA_pos = get_significant_GeO_scores(results_dir, 'pos')
pncA_neg = get_significant_GeO_scores(results_dir, 'neg')

# flip the sign for negative scores because we did that before
if pncA_neg is not None:
    pncA_neg['average'] *= -1

pncA_combined_df = pd.concat([pncA_pos, pncA_neg]).drop_duplicates('residue').sort_values("residue")
pncA_combined_df.to_csv("../supplement/pncA_GeO_scores.csv", index=False)


25 hot spot residues
14 cold spot residues

There are no negative sites


In [83]:
pncA_combined_df.query("residue in @catalytic_triad")

,residue,G_score,average,pval,BH_pval,Bonferroni_pval,prefix
6,8,40.536468,3.841517,0.0050,0.027879,0.9200,pos
94,96,47.370205,0.450679,0.0004,0.010222,0.0736,pos
136,138,51.840115,0.869716,0.0005,0.010222,0.0920,pos


In [84]:
pncA_combined_df.query("residue in @iron_coordinating")

,residue,G_score,average,pval,BH_pval,Bonferroni_pval,prefix
47,49,36.820866,2.248431,0.0076,0.036800,1.0000,pos
49,51,29.757656,3.865622,0.0242,0.072997,1.0000,pos
55,57,55.823019,3.464278,0.0003,0.010222,0.0552,pos
69,71,33.635765,2.208331,0.0133,0.056873,1.0000,pos


In [62]:
results_dir = "katG"

# Arg104, Trp107, and His108 in a pocket distal to the heme, and His270, Trp321, and Asp381 in a pocket proximal to the heme. A covalently linked “MYW catalytic triad” is formed by the conserved residues, Met255, Tyr229, and Trp107

katG_pos = get_significant_GeO_scores(results_dir, 'pos')
katG_neg = get_significant_GeO_scores(results_dir, 'neg')

# flip the sign for negative scores because we did that before
if katG_neg is not None:
    katG_neg['average'] *= -1

katG_combined_df = pd.concat([katG_pos, katG_neg]).drop_duplicates('residue').sort_values("residue")
katG_combined_df.to_csv("../supplement/katG_GeO_scores.csv", index=False)


105 hot spot residues
206 cold spot residues

There are no negative sites


In [117]:
katG_combined_df.query("residue in [104, 107, 108]")

,residue,G_score,average,pval,BH_pval,Bonferroni_pval,prefix
80,104,145.006720,0.410903,0.0038,0.016590,1.0000,pos
83,107,125.432976,0.424340,0.0055,0.020836,1.0000,pos
84,108,207.346162,1.440520,0.0001,0.003768,0.0716,pos


In [119]:
katG_combined_df.query("residue in [270, 321, 381, 255, 229]")

,residue,G_score,average,pval,BH_pval,Bonferroni_pval,prefix
205,229,55.767024,0.469336,0.0372,0.073578,1.0,pos
231,255,37.802873,0.489666,0.0786,0.125340,1.0,pos
246,270,70.393641,0.422707,0.0195,0.046079,1.0,pos
297,321,70.208249,0.487517,0.0190,0.045498,1.0,pos
357,381,65.616863,0.548119,0.0238,0.053310,1.0,pos


In [63]:
results_dir = 'ethA'
ethA_pos = get_significant_GeO_scores(results_dir, 'pos')
ethA_neg = get_significant_GeO_scores(results_dir, 'neg')

# flip the sign for negative scores because we did that before
if ethA_neg is not None:
    ethA_neg['average'] *= -1

ethA_combined_df = pd.concat([ethA_pos, ethA_neg]).drop_duplicates('residue').sort_values("residue")
ethA_combined_df.to_csv("../supplement/ethA_GeO_scores.csv", index=False)


38 hot spot residues
0 cold spot residues

0 hot spot residues
0 cold spot residues


In [92]:
st.spearmanr(ethA_neg['G_score'], ethA_neg['average'])

SignificanceResult(statistic=-0.3818496110630942, pvalue=0.00020427865373023668)

In [93]:
st.spearmanr(ethA_pos['G_score'], ethA_pos['average'])

SignificanceResult(statistic=0.38077554287563015, pvalue=5.66939181583153e-15)

From this link: https://www.uniprot.org/uniprotkb/P9WNF9/entry

<ul>
    <li>FAD binding sites: 15, 36, 44-47, 56, 104</li>
    <li>NADP+ binding sites: 54-56, 183-189, 207-208</li>
    <li>Transition state stabilizer: 292</li>
</ul>

In [112]:
# predicted binding site
FAD_binding_sites = [15, 36, 44, 45, 46, 47, 56, 104]
NADP_binding_site = [54, 55, 56, 183, 184, 185, 186, 187, 188, 189, 207, 208]
# transition_state_stabilizer = [292]

ethA_combined_df.loc[ethA_combined_df['residue'].isin(FAD_binding_sites), 'annotation'] = 'FAD binding'
ethA_combined_df.loc[ethA_combined_df['residue'].isin(NADP_binding_site), 'annotation'] = 'NADP+ binding'
# ethA_combined_df.loc[ethA_combined_df['residue'].isin(transition_state_stabilizer), 'annotation'] = 'transition state stabilizer'

ethA_combined_df.query("(residue in @FAD_binding_sites | residue in @NADP_binding_site | residue in @transition_state_stabilizer) & BH_pval <= 0.05").sort_values("G_score", ascending=False)[['residue', 'G_score', 'annotation', 'BH_pval', 'prefix']]

,residue,G_score,annotation,BH_pval,prefix
149,186,121.837431,NADP+ binding,0.003920,pos
152,189,103.162079,NADP+ binding,0.009046,pos
148,185,101.377551,NADP+ binding,0.011760,pos
37,47,93.835115,FAD binding,0.015680,pos
150,187,92.980343,NADP+ binding,0.009046,pos
147,184,92.826504,NADP+ binding,0.011760,pos
151,188,92.717586,NADP+ binding,0.013067,pos
34,44,92.380519,FAD binding,0.019600,pos
146,183,85.207520,NADP+ binding,0.015339,pos
45,55,84.123890,NADP+ binding,0.035636,pos


In [113]:
ethA_combined_df.annotation.value_counts()

annotation
nan              463
NADP+ binding     12
FAD binding        7
Name: count, dtype: int64

In [115]:
# 14/19 predicted binding sites, cool

In [90]:
results_dir = 'gyrA_N_terminus'
gyrA_pos = get_significant_GeO_scores(results_dir, 'pos')
gyrA_neg = get_significant_GeO_scores(results_dir, 'neg')

# flip the sign for negative scores because we did that before
if gyrA_neg is not None:
    gyrA_neg['average'] *= -1

gyrA_combined_df = pd.concat([gyrA_pos, gyrA_neg]).drop_duplicates('residue').sort_values("residue")
gyrA_combined_df.to_csv("../supplement/gyrA_GeO_scores.csv", index=False)


183 hot spot residues
104 cold spot residues

39 hot spot residues
0 cold spot residues


In [94]:
gyrA_combined_df.query("prefix=='neg' & BH_pval <= 0.05 & G_score > 0").sort_values("G_score", ascending=False).residue.min(), gyrA_combined_df.query("prefix=='neg' & BH_pval <= 0.05 & G_score > 0").sort_values("G_score", ascending=False).residue.max()

(449, 487)

In [91]:
gyrA_combined_df.query("prefix=='neg' & BH_pval <= 0.05 & G_score > 0").sort_values("G_score", ascending=False)

,residue,G_score,average,pval,BH_pval,Bonferroni_pval,prefix
13,462,292.293585,-0.133123,0.0000,0.000000,0.0000,neg
10,459,281.611094,-0.180374,0.0000,0.000000,0.0000,neg
20,469,274.479303,-0.158696,0.0000,0.000000,0.0000,neg
17,466,268.639107,-0.105833,0.0000,0.000000,0.0000,neg
16,465,263.289816,-0.126279,0.0000,0.000000,0.0000,neg
12,461,262.789158,-0.153178,0.0000,0.000000,0.0000,neg
24,473,261.568175,-0.154575,0.0000,0.000000,0.0000,neg
6,455,261.358111,-0.124148,0.0000,0.000000,0.0000,neg
11,460,259.233731,-0.179084,0.0000,0.000000,0.0000,neg
7,456,258.805512,-0.285364,0.0000,0.000000,0.0000,neg


In [95]:
def print_pymol_selection_commands(df, pval_thresh=0.05, hot_color='firebrick', cold_color='skyblue', chain_B=False):

    pval_col = 'BH_pval'
    
    # reset the coloring to gray
    print("select all")
    print("color gray80, all\n")

    # hot_spots = [f'A:{num}' for num in ethA_clustering_lineage_amino_acid.query("BH_pval <= @pval_thresh & G_score > 0").residue.values]
    hot_spots = [str(num) for num in df.query(f"{pval_col} <= @pval_thresh & G_score > 0 & prefix=='pos'").residue.values]
    hot_spots = '+'.join(hot_spots)

    if len(hot_spots) > 0:
        print(f"select hot_spots, resi {hot_spots} and chain A")
        print(f"color {hot_color}, hot_spots\n")

        if chain_B:
            print(f"select hot_spots, resi {hot_spots} and chain B")
            print(f"color {hot_color}, hot_spots\n")
        
    # hot_spots = [f'A:{num}' for num in ethA_clustering_lineage_amino_acid.query("BH_pval <= @pval_thresh & G_score > 0").residue.values]
    cold_spots = [str(num) for num in df.query(f"{pval_col} <= @pval_thresh & G_score > 0 & prefix=='neg'").residue.values]
    cold_spots = '+'.join(cold_spots)

    if len(cold_spots) > 0:
        print(f"select cold_spots, resi {cold_spots} and chain A")
        print(f"color {cold_color}, cold_spots\n")

        if chain_B:
            print(f"select cold_spots, resi {cold_spots} and chain B")
            print(f"color {cold_color}, cold_spots\n")

In [97]:
print_pymol_selection_commands(gyrA_combined_df, chain_B=True)

select all
color gray80, all

select hot_spots, resi 2+10+12+13+14+15+16+17+18+19+20+21+22+23+24+25+26+27+28+29+30+31+32+33+34+39+40+41+42+43+44+45+46+47+48+49+50+51+52+53+54+55+56+57+58+59+60+61+62+63+64+65+66+67+68+69+70+71+72+73+74+75+76+77+78+79+80+81+82+83+84+85+86+87+88+89+90+91+92+93+94+95+96+97+98+99+100+101+102+103+104+105+106+107+108+109+110+111+112+113+114+115+116+117+118+119+120+121+122+123+124+125+126+127+128+129+130+131+147+148+149+151+152+154+155+156+164+165+167+168+169+170+171+172+173+174+175+176+177+178+179+180+181+182+183+220+221+222+253+256+259+260+261+262+263+264+265+266+267+268+269+270+271+272+273+274+276+277+289+290+291+292+293+294+295+296+297+298+299+300+301+302+303 and chain A
color firebrick, hot_spots

select hot_spots, resi 2+10+12+13+14+15+16+17+18+19+20+21+22+23+24+25+26+27+28+29+30+31+32+33+34+39+40+41+42+43+44+45+46+47+48+49+50+51+52+53+54+55+56+57+58+59+60+61+62+63+64+65+66+67+68+69+70+71+72+73+74+75+76+77+78+79+80+81+82+83+84+85+86+87+88+89+90+91+92+93+

In [101]:
print_pymol_selection_commands(pncA_combined_df)

select all
color gray80, all

select hot_spots, resi 7+8+49+50+55+56+57+58+59+67+68+96+102+132+134+135+136+137+138+139+140+141+142+143+181 and chain A
color firebrick, hot_spots



In [13]:
print("select catalytic_triad, resi 8+96+138 and chain A")
print("color magenta, catalytic_triad\n")

print("select iron_coordinating, resi 49+51+57+71 and chain A")
print("color cyan, iron_coordinating")

select catalytic_triad, resi 8+96+138 and chain A
color magenta, catalytic_triad

select iron_coordinating, resi 49+51+57+71 and chain A
color cyan, iron_coordinating


In [102]:
print_pymol_selection_commands(ethA_combined_df)

select all
color gray80, all

select hot_spots, resi 44+45+46+47+48+49+50+52+53+54+55+56+163+164+165+166+167+183+184+185+186+187+188+189+190+191+193+295+296+298+299+300+301+302+304+341+342+344 and chain A
color firebrick, hot_spots



In [6]:
print_pymol_selection_commands(katG_combined_df, chain_B=True)

select all
color gray80, all

select hot_spots, resi 87+88+89+90+91+92+93+94+95+96+97+98+99+100+101+102+103+104+105+106+107+108+109+110+111+112+113+120+121+122+123+124+125+126+127+128+129+132+133+134+135+136+137+138+139+140+141+142+143+144+145+146+147+148+149+161+162+165+166+228+230+231+232+233+263+265+266+270+273+274+275+276+277+278+284+288+297+298+299+300+301+302+307+308+309+310+311+312+313+314+315+316+317+318+319+321+326+367+368+378+379+415+418+419+420 and chain A
color firebrick, hot_spots

select hot_spots, resi 87+88+89+90+91+92+93+94+95+96+97+98+99+100+101+102+103+104+105+106+107+108+109+110+111+112+113+120+121+122+123+124+125+126+127+128+129+132+133+134+135+136+137+138+139+140+141+142+143+144+145+146+147+148+149+161+162+165+166+228+230+231+232+233+263+265+266+270+273+274+275+276+277+278+284+288+297+298+299+300+301+302+307+308+309+310+311+312+313+314+315+316+317+318+319+321+326+367+368+378+379+415+418+419+420 and chain B
color firebrick, hot_spots



In [32]:
katG_clustering_amino_acid.query("residue > 640 & residue < 725")

,residue,G_score,average,pval,BH_pval,Bonferroni_pval
617,641,-49.762245,0.016206,0.0001,0.000770,0.0716
618,642,-53.016117,0.022470,0.0000,0.000000,0.0000
619,643,-54.304159,0.017522,0.0001,0.000770,0.0716
620,644,-55.777549,0.008993,0.0000,0.000000,0.0000
621,645,-56.772647,0.005481,0.0000,0.000000,0.0000
...,...,...,...,...,...,...
696,720,-54.546557,-0.025827,0.0000,0.000000,0.0000
697,721,-52.027264,0.005250,0.0001,0.000770,0.0716
698,722,-44.078390,-0.018913,0.0005,0.003255,0.3580
699,723,-44.079597,-0.001353,0.0010,0.006017,0.7160


In [11]:
print_pymol_selection_commands(ethA_clustering_lineage_amino_acid, pval_thresh=0.05)

select all
color gray80, all

select hot_spots, resi 44+45+46+47+48+49+50+51+52+53+54+55+56+57+62+75+78+146+147+148+162+163+164+165+166+167+180+181+182+183+184+185+186+187+188+189+190+191+206+210+212+292+293+294+295+339+340+341+342+343+344+390+391+438 and chain A
color firebrick, hot_spots

select cold_spots, resi 2+3+4+5+6+7+29+30+31+32+33+96+97+113+115+129+130+131+132+479 and chain A
color skyblue, cold_spots


In [ ]:
print_pymol_selection_commands(katG_clustering_amino_acid, pval_thresh=0.05, chain_B=True)